Settings

In [ ]:
import torch.nn as nn
import torch
import os

DATASET = "all_cards"
num_workers = os.cpu_count() // 2
batch_size = 256
criterion = nn.BCEWithLogitsLoss()

# Device choice
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(torch.__version__)
print(torch.version.cuda)

Read data manifest

In [ ]:
import pandas as pd

df = pd.read_csv(f"data/{DATASET}_manifest.csv")
print(f"Total cards: {len(df)}")
print(df.head())
print(df["types"].value_counts().head(5))

Build dataset

In [ ]:
from collections import Counter

all_types = sorted(set(
    t for types_str in df["types"] if isinstance(types_str, str) for t in types_str.strip().split("|")
))
print(all_types)
print(f"amount of creature types:", len(all_types))

type_counts = Counter(
    t for types_str in df["types"]
    if isinstance(types_str, str)
    for t in types_str.split("|")
)
print(type_counts)

import matplotlib.pyplot as plt

types_sorted = sorted(type_counts.items(), key=lambda x: x[1], reverse=True)
labels, counts = zip(*types_sorted)

print(types_sorted)

plt.figure(figsize=(20, 6))
plt.bar(labels, counts)
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

Data augmentation

In [ ]:
from src.dataset import show_transform

show_transform(x=5, y=3)

Data split

In [ ]:
import random
from src.dataset import CreatureDataset
from src.semantics import get_smoothing_matrix

# create a fixed train/val split
indices = list(range(len(df)))
split = int(0.8 * len(df))
random.shuffle(indices)

train_indices = indices[:split]
val_indices = indices[split:]

smoothing_matrix = get_smoothing_matrix(all_types)

train_dataset = CreatureDataset(df.iloc[train_indices], all_types, is_training=True, smoothing=smoothing_matrix)
val_dataset = CreatureDataset(df.iloc[val_indices], all_types, is_training=False, smoothing=smoothing_matrix)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Building non-uniform sampler

In [ ]:
from src.sampler import get_sampler
from torch.utils.data import DataLoader

sampler = get_sampler(df.iloc[train_indices], alpha=-0.25)

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    sampler=sampler, 
    num_workers=num_workers,
    pin_memory=device == "cuda",
    persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=num_workers,
    pin_memory=device == "cuda",
    persistent_workers=True
)

Training setup

In [ ]:
# Torch settings
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.enabled = False
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_num_threads(os.cpu_count())
torch.backends.mkldnn.enabled = True  # usually on by default, but worth verifying

Run training

In [ ]:
from src.models.efficientnet_b2 import build_efficientnet_b2
from src.train import train

en_b2_config = build_efficientnet_b2(types=all_types, device=device)
train(
    config=en_b2_config, 
    device=device, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    criterion=criterion, 
    types=all_types
)